# SQL-Mode Capability Showcase

This notebook demonstrates every Gremlin step that is supported in SQL-translation mode
(`/query/explain`), as documented in **Section 17** of `DESIGN_DOCUMENT.md`.

Each section shows:
- The Gremlin traversal
- The translated SQL (via `/query/explain`)
- Live results from native execution (via `/gremlin/query`)

**Prerequisites:**
1. Backend running: `mvn exec:java` (in repo root)
2. AML demo data loaded (run the setup cell below, or use `aml_demo_queries.ipynb` Step 2)

---

**Data model** (AML mapping):
```
Account --[TRANSFER]--> Account
Account --[BELONGS_TO]--> Bank
Account --[FLAGGED_BY]--> Alert
Account --[SENT_VIA]--> Country
Bank    --[LOCATED_IN]--> Country
```

In [ ]:
import requests
import json as _json
import pandas as pd
from IPython.display import display, Markdown
from pathlib import Path

BASE_URL = "http://localhost:7000"
MAPPING_PATH = next(
    (str(p) for p in [
        Path.cwd() / "mappings/aml-mapping.json",
        Path.cwd() / "demo/mappings/aml-mapping.json",
    ] if p.exists()),
    None,
)


def explain(gremlin: str) -> dict:
    """Call /query/explain and return the parsed JSON."""
    try:
        r = requests.post(
            f"{BASE_URL}/query/explain",
            json={"gremlin": gremlin},
            timeout=10,
        )
        return r.json() if r.ok else {"error": f"HTTP {r.status_code}: {r.text}"}
    except Exception as e:
        return {"error": str(e)}


def run(gremlin: str) -> dict:
    """Call /gremlin/query and return the parsed JSON."""
    try:
        r = requests.post(
            f"{BASE_URL}/gremlin/query",
            json={"gremlin": gremlin},
            timeout=30,
        )
        return r.json() if r.ok else {"error": f"HTTP {r.status_code}: {r.text}"}
    except Exception as e:
        return {"error": str(e)}


def show(gremlin: str, title: str = "", limit: int = 10):
    """Render Gremlin, SQL translation and live results side-by-side."""
    if title:
        display(Markdown(f"### {title}"))

    display(Markdown(f"**Gremlin:**\n```groovy\n{gremlin}\n```"))

    # SQL translation
    ex = explain(gremlin)
    if "error" in ex:
        display(Markdown(f"**SQL Translation:** *unavailable \u2014 {ex['error']}*"))
    else:
        sql = ex.get("translatedSql", "")
        params = ex.get("parameters", [])
        if sql:
            display(Markdown(f"**SQL:**\n```sql\n{sql}\n```"))
            if params:
                display(Markdown(f"**Parameters:** `{params}`"))
        else:
            display(Markdown("**SQL Translation:** *empty response*"))

    # Live results
    res = run(gremlin)
    if "error" in res:
        display(Markdown(f"**Runtime Error:** {res['error']}"))
        return

    rows = res.get("results", [])
    display(Markdown(f"**Result count:** {res.get('resultCount', len(rows))}"))
    if rows:
        sample = rows[:limit]
        if isinstance(sample[0], dict):
            display(pd.DataFrame(sample))
        else:
            for i, v in enumerate(sample, 1):
                print(f"{i}. {v}")


def pick_first_non_empty(candidates: list[tuple[str, str]]) -> tuple[str, str, int]:
    """Return the first candidate query with resultCount > 0 (best-effort)."""
    for label, gremlin in candidates:
        res = run(gremlin)
        if "error" in res:
            continue
        rows = res.get("results", [])
        count = int(res.get("resultCount", len(rows)))
        if count > 0:
            return label, gremlin, count

    label, gremlin = candidates[0]
    res = run(gremlin)
    rows = res.get("results", []) if isinstance(res, dict) and "error" not in res else []
    count = int(res.get("resultCount", len(rows))) if isinstance(res, dict) and "error" not in res else 0
    return label, gremlin, count


print("BASE_URL :", BASE_URL)
print("MAPPING  :", MAPPING_PATH)

## Setup — Health check, mapping upload & data load

In [ ]:
# Health
try:
    health = requests.get(f"{BASE_URL}/health", timeout=5).text
    provider = requests.get(f"{BASE_URL}/gremlin/provider", timeout=5).json().get("provider", "?")
    display(Markdown(f"**Health:** `{health}`    **Provider:** `{provider}`"))
except Exception as e:
    display(Markdown(f"**Backend not reachable:** {e}"))

# Mapping upload (idempotent)
if MAPPING_PATH:
    with open(MAPPING_PATH, "rb") as f:
        resp = requests.post(f"{BASE_URL}/mapping/upload", files={"file": f}, timeout=10)
    display(Markdown(f"**Mapping upload:** HTTP {resp.status_code} \u2014 {resp.text[:120]}"))
else:
    display(Markdown("**Mapping file not found.** Ensure `mappings/aml-mapping.json` exists."))


def _count(gremlin: str, default: int = 0) -> int:
    try:
        r = requests.post(
            f"{BASE_URL}/gremlin/query",
            json={"gremlin": gremlin},
            timeout=10,
        ).json()
        vals = r.get("results") or []
        return int(vals[0]) if vals else default
    except Exception:
        return default


account_count              = _count("g.V().hasLabel('Account').count()")
high_risk_count            = _count("g.V().hasLabel('Account').has('riskScore', gt(70)).count()")
opened_missing_count       = _count("g.V().hasLabel('Account').hasNot('openedDate').count()")
blocked_count              = _count("g.V().hasLabel('Account').has('isBlocked','true').count()")
suspicious_unflagged_count = _count("g.V().hasLabel('Account').where(and(outE('TRANSFER').has('isLaundering','1'), outE('FLAGGED_BY').count().is(0))).count()")

# Reload when dataset is empty OR appears to be legacy loader output.
needs_reload = account_count == 0 or (
    account_count > 0
    and high_risk_count == 0
    and opened_missing_count == 0
    and blocked_count == 0
    and suspicious_unflagged_count == 0
)

if needs_reload:
    reason = "empty graph" if account_count == 0 else "legacy dataset detected"
    display(Markdown(f"**Data loading:** {reason} \u2014 loading AML demo data..."))
    try:
        from demo.aml_csv_loader import AmlCsvLoader

        loader   = AmlCsvLoader(base_url=BASE_URL)
        csv_path = next(
            (str(p) for p in [
                Path.cwd() / "demo/data/aml-demo.csv",
                Path.cwd() / "data/aml-demo.csv",
            ] if p.exists()),
            None,
        )
        if csv_path:
            stats = loader.load(csv_path, max_rows=100000, verbose=False)
            account_count              = _count("g.V().hasLabel('Account').count()")
            high_risk_count            = _count("g.V().hasLabel('Account').has('riskScore', gt(70)).count()")
            opened_missing_count       = _count("g.V().hasLabel('Account').hasNot('openedDate').count()")
            blocked_count              = _count("g.V().hasLabel('Account').has('isBlocked','true').count()")
            suspicious_unflagged_count = _count("g.V().hasLabel('Account').where(and(outE('TRANSFER').has('isLaundering','1'), outE('FLAGGED_BY').count().is(0))).count()")
            display(Markdown(
                f"\u2713 **Data loaded successfully:**\n"
                f"- Rows loaded: {stats.get('rowsLoaded', 0):,}\n"
                f"- Accounts created: {stats.get('accountsCreated', 0):,}\n"
                f"- Transfers created: {stats.get('transfersCreated', 0):,}\n"
                f"- Accounts in graph: {account_count:,}\n"
                f"- High risk (>70): {high_risk_count:,}\n"
                f"- Blocked accounts: {blocked_count:,}\n"
                f"- Missing openedDate: {opened_missing_count:,}\n"
                f"- Suspicious but unflagged: {suspicious_unflagged_count:,}"
            ))
        else:
            display(Markdown("\u2717 **CSV file not found.** Place `aml-demo.csv` in `demo/data/` or `data/`."))
    except ImportError:
        display(Markdown("\u26a0\ufe0f **Demo module not available.** Run from repo root with Python path configured."))
    except Exception as e:
        display(Markdown(f"\u2717 **Data load failed:** {str(e)[:200]}"))
else:
    display(Markdown(
        f"\u2713 **Data already loaded:**\n"
        f"- Accounts: {account_count:,}\n"
        f"- High risk (>70): {high_risk_count:,}\n"
        f"- Blocked accounts: {blocked_count:,}\n"
        f"- Missing openedDate: {opened_missing_count:,}\n"
        f"- Suspicious but unflagged: {suspicious_unflagged_count:,}"
    ))

# Sanity check translator support for the section 9d predicate.
check_9d = explain("g.V().hasLabel('Account').where(outE('FLAGGED_BY').count().is(0)).limit(1)")
if check_9d.get("error"):
    display(Markdown(f"\u26a0\ufe0f **Translator check:** section 9d predicate still unsupported \u2014 {check_9d['error']}"))

---
## 1 — Root steps: `g.V()`, `g.E()`, `g.V(id)`

The three root forms that start every traversal.

In [ ]:
show(
    "g.V().hasLabel('Account').count()",
    title="1a  g.V() — count all Account vertices",
)

In [ ]:
show(
    "g.E().hasLabel('TRANSFER').count()",
    title="1b  g.E() — count all TRANSFER edges",
)

In [ ]:
# Fetch one account ID first so the g.V(id) demo is self-contained
_first_id_res = run("g.V().hasLabel('Account').values('accountId').limit(1)")
_first_account_id = (_first_id_res.get("results") or ["ACC-001"])[0]
show(
    f"g.V().hasLabel('Account').has('accountId','{_first_account_id}').project('accountId','bankId','riskScore').by('accountId').by('bankId').by('riskScore')",
    title="1c  g.V(id) — look up a specific account by property",
)

---
## 2 — Filter steps: `has`, `hasLabel`, `hasId`, `hasNot`, `is`

In [ ]:
show(
    "g.V().hasLabel('Account').has('isBlocked', 'true').project('accountId','bankId','isBlocked').by('accountId').by('bankId').by('isBlocked').limit(10)",
    title="2a  has(k, v) — blocked accounts",
)

In [ ]:
show(
    "g.V().hasLabel('Account').has('riskScore', gt(70)).project('accountId','bankId','riskScore').by('accountId').by('bankId').by('riskScore').order().by(select('riskScore'), Order.desc).limit(10)",
    title="2b  has(k, gt(v)) — high risk-score accounts (predicate filter)",
)

In [ ]:
# hasId — filter vertex by its graph ID (typed literal)
_id_res = run("g.V().hasLabel('Account').id().limit(1)")
_vid = (_id_res.get("results") or [1])[0]
if isinstance(_vid, (int, float)) or (isinstance(_vid, str) and _vid.isdigit()):
    _vid_literal = str(_vid)
else:
    _vid_literal = f"'{_vid}'"
show(
    f"g.V().hasLabel('Account').hasId({_vid_literal}).project('accountId','bankId').by('accountId').by('bankId')",
    title="2c  hasId(id) — look up vertex by its graph ID",
)

In [ ]:
show(
    "g.V().hasLabel('Account').hasNot('openedDate').project('accountId','bankId').by('accountId').by('bankId').limit(10)",
    title="2d  hasNot(k) — accounts with no openedDate (IS NULL)",
)

In [ ]:
show(
    "g.V().hasLabel('Account').values('riskScore').is(gt(80)).limit(10)",
    title="2e  values('p').is(pred) — risk scores above 80",
)

---
## 3 — Traversal / hop steps: `out`, `in`, `both`, `outE`, `inE`, `bothE`, `outV`, `inV`, `bothV`, `otherV`

In [ ]:
show(
    "g.V().hasLabel('Account').limit(5).out('BELONGS_TO').project('bankId','bankName').by('bankId').by('bankName')",
    title="3a  out(l) — Account → Bank hop",
)

In [ ]:
show(
    "g.V().hasLabel('Bank').limit(5).in('BELONGS_TO').project('accountId','bankId').by('accountId').by('bankId')",
    title="3b  in(l) — Bank ← Account hop",
)

In [ ]:
show(
    "g.V().hasLabel('Account').limit(3).both('TRANSFER').dedup().project('accountId','bankId').by('accountId').by('bankId').limit(10)",
    title="3c  both(l) — bidirectional TRANSFER neighbours",
)

In [ ]:
show(
    "g.V().hasLabel('Account').limit(5).outE('TRANSFER').project('transactionId','amount','currency','isLaundering').by('transactionId').by('amount').by('currency').by('isLaundering').limit(10)",
    title="3d  outE(l) — outgoing TRANSFER edges",
)

In [ ]:
show(
    "g.V().hasLabel('Account').limit(5).inE('TRANSFER').project('transactionId','amount','currency').by('transactionId').by('amount').by('currency').limit(10)",
    title="3e  inE(l) — incoming TRANSFER edges",
)

In [ ]:
show(
    "g.V().hasLabel('Account').limit(3).bothE('TRANSFER').project('transactionId','amount').by('transactionId').by('amount').limit(10)",
    title="3f  bothE(l) — all TRANSFER edges in either direction (hop step)",
)

In [ ]:
show(
    "g.E().hasLabel('TRANSFER').limit(5).outV().project('accountId','bankId').by('accountId').by('bankId')",
    title="3g  outV() — source vertex of each TRANSFER edge",
)

In [ ]:
show(
    "g.E().hasLabel('TRANSFER').limit(5).inV().project('accountId','bankId').by('accountId').by('bankId')",
    title="3h  inV() — destination vertex of each TRANSFER edge",
)

In [ ]:
show(
    "g.E().hasLabel('TRANSFER').limit(5).bothV().dedup().project('accountId','bankId').by('accountId').by('bankId')",
    title="3i  bothV() — all endpoint vertices of TRANSFER edges (UNION of out+in)",
)

In [ ]:
show(
    "g.V().hasLabel('Account').limit(5).outE('BELONGS_TO').otherV().project('bankId','bankName').by('bankId').by('bankName')",
    title="3j  otherV() — opposite endpoint after outE",
)

---
## 4 — Repeat / path steps

In [ ]:
show(
    "g.V().hasLabel('Account').where(outE('TRANSFER').has('isLaundering','1')).limit(1).repeat(out('TRANSFER').simplePath()).times(2).path().by('accountId').limit(10)",
    title="4a  repeat(out).times(n) + simplePath() — two-hop money trail",
)

In [ ]:
show(
    "g.V().hasLabel('Account').limit(1).repeat(out('BELONGS_TO','LOCATED_IN').simplePath()).times(2).path().by('accountId').by('bankName').by('countryName').limit(10)",
    title="4b  repeat with multi-label + path().by(p) — Account → Bank → Country",
)

---
## 5 — Aggregation steps: `count`, `sum`, `mean`, `groupCount`

In [ ]:
show(
    "g.V().hasLabel('Account').count()",
    title="5a  count() — total Account vertices",
)

In [ ]:
show(
    "g.V().hasLabel('Account').dedup().count()",
    title="5b  dedup() + count() → COUNT(DISTINCT id)",
)

In [ ]:
show(
    "g.E().hasLabel('TRANSFER').values('amount').sum()",
    title="5c  values('amount').sum() — total transferred amount",
)

In [ ]:
show(
    "g.E().hasLabel('TRANSFER').values('amount').mean()",
    title="5d  values('amount').mean() — average transfer amount",
)

In [ ]:
show(
    "g.V().hasLabel('Account').groupCount().by('bankId').order().by(values, desc).limit(10)",
    title="5e  groupCount().by(p) — accounts per bank, descending",
)

---
## 6 — Projection steps: `values`, `project().by()`, `valueMap`, `by(identity())`

In [ ]:
show(
    "g.V().hasLabel('Account').values('accountId').limit(10)",
    title="6a  values(p) — single property column",
)

In [ ]:
show(
    "g.V().hasLabel('Account').project('accountId','bankId','riskScore','accountType').by('accountId').by('bankId').by('riskScore').by('accountType').limit(10)",
    title="6b  project().by(p) — multi-column SELECT",
)

In [ ]:
show(
    "g.V().hasLabel('Account').valueMap().limit(5)",
    title="6c  valueMap() — all mapped properties as a map (root vertex)",
)

In [ ]:
show(
    "g.E().hasLabel('TRANSFER').valueMap().limit(5)",
    title="6d  valueMap() — all mapped properties for edges",
)

In [ ]:
show(
    "g.V().hasLabel('Account').project('id','accountId','bankId').by(identity()).by('accountId').by('bankId').limit(10)",
    title="6e  by(identity()) in project() — emits the vertex/element id",
)

In [ ]:
show(
    "g.V().hasLabel('Account').limit(10).project('accountId','bankId','bankName').by('accountId').by('bankId').by(out('BELONGS_TO').values('bankName').fold())",
    title="6f  by(out(l).values(p).fold()) — neighbour property list per vertex",
)

In [ ]:
show(
    "g.V().hasLabel('Account').project('accountId','outDegree','inDegree').by('accountId').by(outE('TRANSFER').count()).by(inE('TRANSFER').count()).order().by(select('outDegree'), Order.desc).limit(10)",
    title="6g  by(outE/inE.count()) — edge-degree subqueries in project()",
)

In [ ]:
show(
    "g.V().hasLabel('Account').project('accountId','riskBucket').by('accountId').by(choose(values('riskScore').is(gte(70)), constant('HIGH'), constant('NORMAL'))).limit(15)",
    title="6h  by(choose(values.is, constant, constant)) — CASE WHEN projection",
)

In [ ]:
show(
    "g.E().hasLabel('TRANSFER').limit(5).project('fromAccountId','toAccountId','amount').by(outV().values('accountId')).by(inV().values('accountId')).by('amount')",
    title="6i  by(outV/inV.values(p)) — edge project with endpoint properties",
)

---
## 7 — Ordering and paging: `order().by`, `limit`, `dedup`

In [ ]:
show(
    "g.V().hasLabel('Account').order().by('riskScore', Order.desc).project('accountId','bankId','riskScore').by('accountId').by('bankId').by('riskScore').limit(10)",
    title="7a  order().by(p, desc) — top-risk accounts",
)

In [ ]:
show(
    "g.V().hasLabel('Account').project('accountId','outDegree').by('accountId').by(outE('TRANSFER').count()).order().by(select('outDegree'), Order.desc).limit(10)",
    title="7b  order().by(select(a), desc) — order by projected alias",
)

In [ ]:
show(
    "g.V().hasLabel('Account').dedup().count()",
    title="7c  dedup() — distinct accounts (COUNT DISTINCT)",
)

---
## 8 — Alias / select / where steps

In [ ]:
show(
    "g.V().hasLabel('Account').as('a').out('TRANSFER').as('b').where('a', neq('b')).select('a','b').by('accountId').by('accountId').limit(10)",
    title="8a  as(l) + where(a, neq(b)) + select().by(p) — self-loop exclusion",
)

In [ ]:
show(
    "g.V().hasLabel('Account').where(outE('TRANSFER').has('isLaundering','1')).project('accountId','bankId').by('accountId').by('bankId').limit(10)",
    title="8b  where(outE(l).has(k,v)) — correlated EXISTS subquery",
)

In [ ]:
show(
    "g.V().hasLabel('Account').where(inE('TRANSFER').has('isLaundering','1')).project('accountId','bankId').by('accountId').by('bankId').limit(10)",
    title="8c  where(inE(l).has(k,v)) — correlated EXISTS on incoming edge",
)

In [ ]:
show(
    "g.V().hasLabel('Account').where(bothE('TRANSFER').has('isLaundering','1')).project('accountId','bankId').by('accountId').by('bankId').limit(10)",
    title="8d  where(bothE(l).has(k,v)) — EXISTS on either direction",
)

In [ ]:
show(
    "g.V().hasLabel('Account').where(out('SENT_VIA').has('fatfBlacklist','true')).project('accountId','bankId','riskScore').by('accountId').by('bankId').by('riskScore').limit(10)",
    title="8e  where(out(l).has(k,v)) — correlated EXISTS with vertex JOIN",
)

In [ ]:
show(
    "g.V().hasLabel('Account').project('accountId','suspOut').by('accountId').by(outE('TRANSFER').has('isLaundering','1').count()).where(select('suspOut').is(gt(0))).order().by(select('suspOut'), Order.desc).limit(10)",
    title="8f  where(select(a).is(gt(n))) — HAVING-style filter on projected alias",
)

---
## 9 — Compound `where`: `and`, `or`, `not`

In [ ]:
show(
    "g.V().hasLabel('Account').where(and(outE('TRANSFER').has('isLaundering','1'), outE('FLAGGED_BY'))).project('accountId','bankId').by('accountId').by('bankId').limit(10)",
    title="9a  where(and(p1, p2)) — accounts that are both suspicious AND flagged",
)

In [ ]:
show(
    "g.V().hasLabel('Account').where(or(out('SENT_VIA').has('fatfBlacklist','true'), outE('TRANSFER').has('isLaundering','1'))).project('accountId','bankId','riskScore').by('accountId').by('bankId').by('riskScore').limit(10)",
    title="9b  where(or(p1, p2)) — accounts linked to blacklisted country OR suspicious transfer",
)

In [ ]:
show(
    "g.V().hasLabel('Account').where(outE('FLAGGED_BY').count().is(0)).project('accountId','bankId').by('accountId').by('bankId').limit(10)",
    title="9c  where(outE.count().is(0)) — accounts with NO alert flags",
)

In [ ]:
q9d_candidates = [
    ("suspicious but unflagged", "g.V().hasLabel('Account').where(and(outE('TRANSFER').has('isLaundering','1'), outE('FLAGGED_BY').count().is(0))).project('accountId','bankId').by('accountId').by('bankId').limit(10)"),
    ("high-risk and unflagged", "g.V().hasLabel('Account').where(and(has('riskScore', gt(70)), outE('FLAGGED_BY').count().is(0))).project('accountId','bankId').by('accountId').by('bankId').limit(10)"),
    ("active and unflagged", "g.V().hasLabel('Account').where(and(outE('TRANSFER').count().is(gt(0)), outE('FLAGGED_BY').count().is(0))).project('accountId','bankId').by('accountId').by('bankId').limit(10)"),
]
q9d_label, q9d_query, q9d_count = pick_first_non_empty(q9d_candidates)
show(
    q9d_query,
    title=f"9d  where(and(p1, p2)) — {q9d_label} (auto-selected for non-zero demo; count={q9d_count})",
)

---
## 10 — `groupCount` with alias key

In [ ]:
show(
    "g.V().hasLabel('Account').as('a').out('BELONGS_TO').as('b').groupCount().by(select('a').by('bankId')).order().by(values, desc).limit(10)",
    title="10a  groupCount().by(select(a).by(p)) — accounts per bank via alias key",
)

---
## 11 — Edge-root projections with `outV` / `inV`

In [ ]:
show(
    "g.E().hasLabel('TRANSFER').has('isLaundering','1').project('fromAcct','toAcct','amount','currency').by(outV().values('accountId')).by(inV().values('accountId')).by('amount').by('currency').limit(15)",
    title="11a  g.E().project(by(outV.values), by(inV.values)) — suspicious transfer endpoints",
)

---
## 12 — `simplePath` across multi-hop traversals

In [ ]:
show(
    "g.V().hasLabel('Account').where(outE('TRANSFER').has('isLaundering','1')).limit(1).repeat(out('TRANSFER').simplePath()).times(3).path().by('accountId').limit(10)",
    title="12a  simplePath() — 3-hop money trail with cycle exclusion",
)

---
## 13 — `identity()` as no-op step and in `project`

In [ ]:
show(
    "g.V().hasLabel('Account').identity().project('accountId','bankId').by('accountId').by('bankId').limit(5)",
    title="13a  identity() as a no-op mid-traversal step",
)

In [ ]:
show(
    "g.V().hasLabel('Account').project('graphId','accountId','bankId').by(identity()).by('accountId').by('bankId').limit(5)",
    title="13b  project(...).by(identity()) — emit the graph element ID",
)

---
## 14 — Full AML showcase query (combines multiple features)

In [ ]:
q14_predicate_candidates = [
    ("suspicious-but-unflagged", "and(outE('TRANSFER').has('isLaundering','1'), outE('FLAGGED_BY').count().is(0))"),
    ("high-risk-and-unflagged", "and(has('riskScore', gt(70)), outE('FLAGGED_BY').count().is(0))"),
    ("active-and-unflagged", "and(outE('TRANSFER').count().is(gt(0)), outE('FLAGGED_BY').count().is(0))"),
]

q14_candidates = []
for label, predicate in q14_predicate_candidates:
    gremlin = (
        "g.V().hasLabel('Account')"
        f".where({predicate})"
        ".project('accountId','bankId','riskScore','suspiciousOut','totalOut')"
        ".by('accountId')"
        ".by('bankId')"
        ".by('riskScore')"
        ".by(outE('TRANSFER').has('isLaundering','1').count())"
        ".by(outE('TRANSFER').count())"
        ".order().by(select('suspiciousOut'), Order.desc)"
        ".limit(20)"
    )
    q14_candidates.append((label, gremlin))

q14_label, q14_query, q14_count = pick_first_non_empty(q14_candidates)
show(
    q14_query,
    title=f"14  Full AML showcase — {q14_label} accounts with degrees (auto-selected; count={q14_count})",
    limit=20,
)

---

## Summary

| Section | Steps demonstrated |
|---------|--------------------|
| 1 | `g.V()`, `g.E()`, `g.V(id)` — root steps |
| 2 | `has`, `hasLabel`, `hasId`, `hasNot`, `is(pred)` — filter steps |
| 3 | `out`, `in`, `both`, `outE`, `inE`, `bothE`, `outV`, `inV`, `bothV`, `otherV` — hop steps |
| 4 | `repeat().times(n)`, `simplePath()`, `path().by(p)` — path steps |
| 5 | `count`, `sum`, `mean`, `groupCount().by(p)`, `dedup` — aggregation |
| 6 | `values(p)`, `project().by(p)`, `valueMap()`, `by(identity())`, `by(out.values.fold)`, `by(outE/inE.count)`, `by(choose)`, `by(outV/inV.values)` — projection |
| 7 | `order().by(p, asc/desc)`, `order().by(select(a))`, `limit(n)` — ordering/paging |
| 8 | `as`, `select().by`, `where(neq)`, `where(outE/inE/bothE.has)`, `where(out.has)`, `where(select.is(gt))` — alias/where |
| 9 | `where(and(...))`, `where(or(...))`, `where(not(...))` — compound predicates |
| 10 | `groupCount().by(select(a).by(p))` — alias-keyed grouping |
| 11 | `g.E().project(by(outV.values), by(inV.values))` — edge-root projections |
| 12 | `simplePath()` across multi-hop `repeat` |
| 13 | `identity()` as no-op and in `project().by(identity())` |
| 14 | Full combined query |

All queries in sections 1–13 are translated to SQL by `/query/explain`.
Native execution (`/gremlin/query`) is used to show live results.